In [0]:
WITH source_counts AS (
  SELECT
    year,
    month,
    COUNT(*) AS source_count
  FROM aws_glue_nyc_taxi.nyc_taxi_study.yellow_tripdata
  WHERE year = '2025'
  GROUP BY year, month
),

bronze_counts AS (
  SELECT
    year,
    month,
    COUNT(*) AS bronze_count
  FROM nyc_taxi.bronze.bronze_yellow_trip_2025
  WHERE year = '2025'
  GROUP BY year, month
),

comparison AS (
  SELECT
    COALESCE(s.year, b.year) AS year,
    COALESCE(s.month, b.month) AS month,
    COALESCE(s.source_count, 0) AS source_count,
    COALESCE(b.bronze_count, 0) AS bronze_count
  FROM source_counts AS s
  FULL OUTER JOIN bronze_counts AS b
    ON s.year = b.year
   AND s.month = b.month
)

SELECT
  COUNT(*) AS compared_partitions,
  SUM(source_count) AS source_total,
  SUM(bronze_count) AS bronze_total,
  SUM(bronze_count) - SUM(source_count) AS total_difference,
  COUNT_IF(source_count <> bronze_count) AS divergent_partitions,
  CASE
    WHEN COUNT(*) = 12
     AND COUNT_IF(source_count <> bronze_count) = 0
      THEN 'BRONZE_VALIDATED'
    ELSE 'VALIDATION_FAILED'
  END AS validation_status
FROM comparison;

WITH location_summary AS (
  SELECT
    LocationID,
    COUNT(*) AS occurrences
  FROM nyc_taxi.bronze.bronze_taxi_zone_lookup
  GROUP BY LocationID
),

duplicate_summary AS (
  SELECT
    COUNT_IF(occurrences > 1) AS duplicated_location_ids,
    COALESCE(
      SUM(
        CASE
          WHEN occurrences > 1 THEN occurrences - 1
          ELSE 0
        END
      ),
      0
    ) AS duplicated_rows
  FROM location_summary
),

profile AS (
  SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT LocationID) AS distinct_location_ids,

    COUNT_IF(LocationID IS NULL) AS null_location_ids,

    COUNT_IF(
      Borough IS NULL
      OR TRIM(Borough) = ''
    ) AS missing_borough,

    COUNT_IF(
      Zone IS NULL
      OR TRIM(Zone) = ''
    ) AS missing_zone,

    COUNT_IF(
      service_zone IS NULL
      OR TRIM(service_zone) = ''
    ) AS missing_service_zone,

    COUNT_IF(
      _source_system IS NULL
      OR _source_system <> 'unity_catalog_volume'
    ) AS invalid_source_system,

    COUNT_IF(
      _source_format IS NULL
      OR _source_format <> 'csv'
    ) AS invalid_source_format,

    COUNT_IF(
      _source_path IS NULL
      OR TRIM(_source_path) = ''
    ) AS missing_source_path,

    COUNT_IF(
      _pipeline_refresh_timestamp IS NULL
    ) AS missing_refresh_timestamp

  FROM nyc_taxi.bronze.bronze_taxi_zone_lookup
)

SELECT
  p.row_count,
  p.distinct_location_ids,
  p.null_location_ids,
  d.duplicated_location_ids,
  d.duplicated_rows,
  p.missing_borough,
  p.missing_zone,
  p.missing_service_zone,
  p.invalid_source_system,
  p.invalid_source_format,
  p.missing_source_path,
  p.missing_refresh_timestamp,

  CASE
    WHEN p.row_count <> 265
      THEN 'UNEXPECTED_ROW_COUNT'

    WHEN p.null_location_ids > 0
      THEN 'NULL_LOCATION_ID'

    WHEN d.duplicated_location_ids > 0
      THEN 'DUPLICATED_LOCATION_ID'

    WHEN p.missing_borough > 0
      OR p.missing_zone > 0
      OR p.missing_service_zone > 0
      THEN 'MISSING_REFERENCE_ATTRIBUTE'

    WHEN p.invalid_source_system > 0
      OR p.invalid_source_format > 0
      OR p.missing_source_path > 0
      OR p.missing_refresh_timestamp > 0
      THEN 'INVALID_TECHNICAL_METADATA'

    ELSE 'BRONZE_LOOKUP_VALIDATED'
  END AS validation_status

FROM profile AS p
CROSS JOIN duplicate_summary AS d;